# Amazon ML Challenge 2026 — Full Pipeline
Runs the bundled project source against the attached private dataset and writes submission artifacts to `/kaggle/working`.


In [ ]:
import base64, io, os, zipfile, sys, shutil
from pathlib import Path
BUNDLE_B64 = '...'
PROJECT = Path('/kaggle/working/amazon_ml_pipeline')
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE_B64))) as zf: zf.extractall(PROJECT)
os.environ['AMAZON_ML_BASE_DIR'] = str(PROJECT)
os.environ['AMAZON_ML_DATASET_DIR'] = '/kaggle/input/amazon-ml-2026-dataset'
sys.path.insert(0, str(PROJECT))
assert (Path(os.environ['AMAZON_ML_DATASET_DIR']) / 'train' / 'train_source1.tsv').exists(), 'Attached dataset layout is missing train/train_source1.tsv'
os.chdir(PROJECT)
import subprocess
subprocess.run([sys.executable, '-m', 'src.run_pipeline', '--stage', 'all'], cwd=PROJECT, check=True)
for name in ('matching_results.tsv', 'candidate_pairs.tsv'):
    shutil.copy2(PROJECT / 'output' / name, Path('/kaggle/working') / name)
for name in ('model.pkl', 'optimal_threshold.json'):
    p = PROJECT / 'artifacts' / 'models' / name
    if p.exists(): shutil.copy2(p, Path('/kaggle/working') / name)
print('Submission outputs written to /kaggle/working')
